# Week 11 - The Old Tongue

This week we used a new dataset of movie reviews. This contained full sentences and a label
for positive and negative review. From this dataset we can get a words sentiment, create
a similarity vector to associate with other words, and even train a predictive model to fill
in blanks between words.

## Part 1 - Preprocess
For preprocessing we tokenize the words from the reviews and we regularize positive and
negative to 0s and 1s. 

In [14]:
import numpy as np
from forms import sigmoid

def open_files():
    with open("data/reviews.txt", "r", encoding="utf-8") as f:
        raw_reviews = f.read().splitlines()

    with open("data/labels.txt", "r", encoding="utf-8") as f:
        raw_labels = f.read().splitlines()

    return raw_reviews, raw_labels

def tokenize_inputs():
    raw_reviews, raw_labels = open_files()

    tokens = list(map(lambda s: set(s.split(" ")), raw_reviews))
    tokens_list = list(map(lambda s: list(s.split(" ")), raw_reviews))

    vocab = set()
    for sent in tokens:
        for word in sent:
            if len(word) > 0: 
                vocab.add(word)
    vocab = sorted(vocab) # sorted: a set's order changes every run

    word2index = {w: i for i, w in enumerate(vocab)}
    input_dataset = [list({word2index[w] for w in sent if w in word2index}) for sent in tokens]
    input_dataset_list = [list([word2index[w] for w in sent if w in word2index]) for sent in tokens_list]

    new_labels = regularize_labels(raw_labels)

    return raw_reviews, raw_labels, new_labels, vocab, input_dataset, input_dataset_list

def regularize_labels(raw_labels):
    new_labels = []
    for label in raw_labels:
        if label.strip().lower() == "positive":
            new_labels.append(1)
        else:
            new_labels.append(0)
    return new_labels

## Part 2 - Sentiment

For this section, we train word sentiment to a positive or negative association. This section will end up overfitting
somewhat. This ends up getting solid training accuracy with a significant dip when it comes to testing accuracy.
We can continue training but no improvement will be made for the testing dataset.

In [15]:
def word_to_index(vocab):
    word2index = {w: i for i, w in enumerate(vocab)}
    return word2index

def sentiment(vocab, new_labels, input_dataset, hidden_size, alpha, iterations, seed = 1, eval_test_size = 1000, verbose = True):
    np.random.seed(seed)

    weights_0_1 = 0.2 * np.random.random((len(vocab), hidden_size)) - 0.1
    weights_1_2 = 0.2 * np.random.random((hidden_size, 1)) - 0.1

    for it in range(0, iterations):
        correct = 0

        for i in range(len(input_dataset) - eval_test_size):
            x, y = input_dataset[i], new_labels[i]

            # Forward Pass
            layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0)) # row-sum lookup
            layer_2 = sigmoid(np.dot(layer_1, weights_1_2))

            # Error Tracking
            prediction = layer_2.item()
            correct += int((prediction >= 0.5) == y)

            # Delta
            layer_2_delta = layer_2 - y
            layer_1_delta = layer_2_delta.dot(weights_1_2.T)
            layer_1_delta *= layer_1 * (1 - layer_1) # sigmoid's slope

            # Weight Updates
            weights_0_1[x] -= layer_1_delta * alpha # only rows for x
            weights_1_2 -= np.outer(layer_1, layer_2_delta) * alpha

            if verbose and (i + 1) % 1000 == 0:
                print(f"Iter: {it + 1}/{iterations}\tReview: {i + 1}\tAccuracy: {correct / (i + 1) * 100:.4f}%")

    word2index = word_to_index(vocab)

    return weights_0_1, weights_1_2, word2index, vocab

def accuracy_testing(weights_0_1, weights_1_2, input_dataset, new_labels, eval_test_size=1000):
    test_correct = 0
    test_start = len(input_dataset) - eval_test_size

    for i in range(test_start, len(input_dataset)):
        x, y = input_dataset[i], new_labels[i]

        layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
        layer_2 = sigmoid(np.dot(layer_1, weights_1_2))

        prediction = layer_2.item()
        test_correct += int((prediction >= 0.5) == y)

    accuracy_testing = test_correct / eval_test_size * 100

    print(f"\nFinal Test Accuracy: {accuracy_testing:.4f}%")

In [16]:
raw_reviews, raw_labels, new_labels, vocab, input_dataset, input_dataset_list = tokenize_inputs()

weights_0_1, weights_1_2, word2index, vocab = sentiment(vocab, new_labels, input_dataset, 100, 0.04, 4)

accuracy_testing(weights_0_1, weights_1_2, input_dataset, new_labels, 1000)

Iter: 1/4	Review: 1000	Accuracy: 42.5000%
Iter: 1/4	Review: 2000	Accuracy: 59.4000%
Iter: 1/4	Review: 3000	Accuracy: 67.2000%
Iter: 1/4	Review: 4000	Accuracy: 70.5250%
Iter: 1/4	Review: 5000	Accuracy: 72.3200%
Iter: 1/4	Review: 6000	Accuracy: 73.9833%
Iter: 1/4	Review: 7000	Accuracy: 75.5714%
Iter: 1/4	Review: 8000	Accuracy: 76.8625%
Iter: 1/4	Review: 9000	Accuracy: 77.7444%
Iter: 1/4	Review: 10000	Accuracy: 78.7200%
Iter: 1/4	Review: 11000	Accuracy: 79.3182%
Iter: 1/4	Review: 12000	Accuracy: 79.8750%
Iter: 1/4	Review: 13000	Accuracy: 80.4154%
Iter: 1/4	Review: 14000	Accuracy: 80.8857%
Iter: 1/4	Review: 15000	Accuracy: 81.0800%
Iter: 1/4	Review: 16000	Accuracy: 81.2375%
Iter: 1/4	Review: 17000	Accuracy: 81.4235%
Iter: 1/4	Review: 18000	Accuracy: 81.7056%
Iter: 1/4	Review: 19000	Accuracy: 81.9737%
Iter: 1/4	Review: 20000	Accuracy: 82.2800%
Iter: 1/4	Review: 21000	Accuracy: 82.5190%
Iter: 1/4	Review: 22000	Accuracy: 82.7318%
Iter: 1/4	Review: 23000	Accuracy: 82.8913%
Iter: 1/4	Review: 24

## Part 3 - Similarity

In this section, we utilize the weights that we trained with to find similarity with the word sentiments.
So not only can we get a positive/negative sentiment prediction, the weights that learn this pattern
can have association between words.

In [17]:
from collections import Counter
def similar(weights_0_1, word2index, target='beautiful'):
    target_vec = weights_0_1[word2index[target]]
    scores = Counter()
    for word, index in word2index.items():
        diff = weights_0_1[index] - target_vec
        scores[word] = -np.sqrt(np.sum(diff * diff)) # smaller dist -> higher score
    return scores.most_common(10)

raw_reviews, raw_labels, new_labels, vocab, input_dataset, input_dataset_list = tokenize_inputs()

weights_0_1, weights_1_2, word2index, vocab = sentiment(vocab, new_labels, input_dataset, 100, 0.04, 2)

common_scores = similar(weights_0_1, word2index)

print("Similar words - beautiful")
for score in common_scores:
    print(f"{score}")

common_scores = similar(weights_0_1, word2index, target="terrible")

print("Similar words - terrible")
for score in common_scores:
    print(f"{score}")

common_scores = similar(weights_0_1, word2index, target="actor")
    
print("Similar words - actor")
for score in common_scores:
    print(f"{score}")

# accuracy_testing(weights_0_1, weights_1_2, input_dataset, new_labels, 1000)

Iter: 1/2	Review: 1000	Accuracy: 42.5000%
Iter: 1/2	Review: 2000	Accuracy: 59.4000%
Iter: 1/2	Review: 3000	Accuracy: 67.2000%
Iter: 1/2	Review: 4000	Accuracy: 70.5250%
Iter: 1/2	Review: 5000	Accuracy: 72.3200%
Iter: 1/2	Review: 6000	Accuracy: 73.9833%
Iter: 1/2	Review: 7000	Accuracy: 75.5714%
Iter: 1/2	Review: 8000	Accuracy: 76.8625%
Iter: 1/2	Review: 9000	Accuracy: 77.7444%
Iter: 1/2	Review: 10000	Accuracy: 78.7200%
Iter: 1/2	Review: 11000	Accuracy: 79.3182%
Iter: 1/2	Review: 12000	Accuracy: 79.8750%
Iter: 1/2	Review: 13000	Accuracy: 80.4154%
Iter: 1/2	Review: 14000	Accuracy: 80.8857%
Iter: 1/2	Review: 15000	Accuracy: 81.0800%
Iter: 1/2	Review: 16000	Accuracy: 81.2375%
Iter: 1/2	Review: 17000	Accuracy: 81.4235%
Iter: 1/2	Review: 18000	Accuracy: 81.7056%
Iter: 1/2	Review: 19000	Accuracy: 81.9737%
Iter: 1/2	Review: 20000	Accuracy: 82.2800%
Iter: 1/2	Review: 21000	Accuracy: 82.5190%
Iter: 1/2	Review: 22000	Accuracy: 82.7318%
Iter: 1/2	Review: 23000	Accuracy: 82.8913%
Iter: 1/2	Review: 24

## Part 4 - Fill in the Blank

In this section, using the surrounding words as context to train on. We use a single correct word,
and 5 incorrect words to negatively weight them. If we did every incorrect word, it would take 
multiplicatively longer to train.

In [18]:
import random
raw_reviews, raw_labels, new_labels, vocab, input_dataset, input_dataset_list = tokenize_inputs()


np.random.seed(1); random.seed(1)
random.shuffle(input_dataset_list) # Trask shuffles the reviews once
alpha, iterations = 0.05, 2
hidden_size, window, neg = 50, 2, 5
weights_0_1 = (np.random.rand(len(vocab), hidden_size) - 0.5) * 0.2
weights_1_2 = np.random.rand(len(vocab), hidden_size) * 0 # zeros
layer_2_target = np.zeros(neg + 1); layer_2_target[0] = 1

def train_fill_blanks(weights_0_1, weights_1_2, dataset, alpha, iterations, window, neg, reassuring = False):
    word2index = word_to_index(vocab)
    concatenated = np.concatenate(dataset)
    for it in range(iterations):
        for i, review in enumerate(dataset):
            if reassuring:
                if i % 1000 == 0:
                    print("Iteration: ", it, "Review: ", i)
                if i % 10000 == 0:
                        print(similar(weights_0_1, word2index, "terrible"))
                        print(similar(weights_0_1, word2index, "beautiful"))
                        print(similar(weights_0_1, word2index, "actor"))
            for t in range(len(review)):
                target_samples = [review[t]] + list(concatenated[(np.random.rand(neg) * len(concatenated)).astype('int')])
                left = review[max(0, t-window): t]
                right = review[t+1: min(len(review), t+window)]
                layer_1 = np.mean(weights_0_1[left + right], axis=0) # (hidden,)
                layer_2 = sigmoid(layer_1.dot(weights_1_2[target_samples].T)) # (6,)
                layer_2_delta = layer_2 - layer_2_target
                layer_1_delta = layer_2_delta.dot(weights_1_2[target_samples])
                weights_0_1[left + right] -= layer_1_delta * alpha
                weights_1_2[target_samples] -= np.outer(layer_2_delta, layer_1) * alpha

    return weights_0_1, weights_1_2

word2index = word_to_index(vocab)
train_fill_blanks(weights_0_1, weights_1_2, input_dataset_list, alpha, iterations, window, neg, reassuring = True)
print(similar(weights_0_1, word2index, "terrible"))
print(similar(weights_0_1, word2index, "beautiful"))
print(similar(weights_0_1, word2index, "actor"))

Iteration:  0 Review:  0
[('terrible', np.float64(-0.0)), ('trubshawe', np.float64(-0.3859867179267425)), ('planes', np.float64(-0.38702859631163194)), ('deplorable', np.float64(-0.39179708187953327)), ('chalon', np.float64(-0.3985233127109086)), ('schenck', np.float64(-0.4007332594340032)), ('prettiness', np.float64(-0.4008459521067322)), ('freeman', np.float64(-0.4037499934182305)), ('desertion', np.float64(-0.4044208990280613)), ('divulges', np.float64(-0.4056383923589251))]
[('beautiful', np.float64(-0.0)), ('cares', np.float64(-0.3621996452208232)), ('layman', np.float64(-0.37395801354325836)), ('okinawan', np.float64(-0.37604763722617135)), ('saito', np.float64(-0.3800914883868123)), ('ricochet', np.float64(-0.38551575005932076)), ('embarrassing', np.float64(-0.39013502214215257)), ('fillum', np.float64(-0.39057390075153187)), ('coilition', np.float64(-0.3910486589976912)), ('dished', np.float64(-0.3915018000236003))]
[('actor', np.float64(-0.0)), ('tarantula', np.float64(-0.3908

## Part 5 - Analogy

In this section, we have a way to show a basic level of "arithmetic" between words to show a
larger level of comparison. This breaks down fairly quickly when trying to use more specific
words. There aren't nearly enough words and uses of them to build a more robust comparison map.

In [19]:
import math
def analogy(positive, negative, weights_0_1, word2index, k=5):
    q = np.zeros(weights_0_1.shape[1])
    for w in positive: q += weights_0_1[word2index[w]]
    for w in negative: q -= weights_0_1[word2index[w]]
    scores = Counter()
    for word, idx in word2index.items():
        d = weights_0_1[idx] - q
        scores[word] = -math.sqrt(np.sum(d * d))
    # exclude the input words from the result
    return [w for w, _ in scores.most_common(20)
            if w not in positive + negative][:k]

labels, vocab, _, dataset = tokenize_inputs()[2:]
word2index = word_to_index(vocab)

np.random.seed(1), random.seed(1)
random.shuffle(dataset)  # Trask shuffles the reviews once
alpha, iterations, hidden_size, window, neg = 0.05, 2, 50, 2, 5
weights_0_1 = (np.random.rand(len(vocab), hidden_size) - 0.5) * 0.2
weights_1_2 = np.random.rand(len(vocab), hidden_size) * 0  # zeros
W_0_1, _ = train_fill_blanks(weights_0_1, weights_1_2, dataset, alpha, iterations, window, neg)

analogies = analogy(
    ['terrible', 'good'],
    ['bad'],
    W_0_1,
    word2index
)
print(f"terrible - bad + good = {analogies}")
"""The results of terrible - bad + good are:
1. fine
2. superb
3. brilliant
4. fantastic
5. wonderful
"""

our_analogies = analogy(
    ['cheery', 'dark'],
    ['light'],
    W_0_1,
    word2index
)
print(f"cheery - light + dark = {our_analogies}")
"""The results of cheery - light + dark are:
1. therein
2. revolutionary
3. monica
4. deol
5. frozen
"""

other_analogy = analogy(
    ['actor', 'woman'],
    ['man'],
    W_0_1,
    word2index
)
print(f"actor - man + woman = {other_analogy}")
"""The results of actor + woman - man are:
1. actress
2. artist
3. antagonist
4. assassin
5. parker
"""

terrible - bad + good = ['fine', 'superb', 'brilliant', 'fantastic', 'wonderful']
cheery - light + dark = ['therein', 'revolutionary', 'monica', 'deol', 'frozen']
actor - man + woman = ['actress', 'artist', 'antagonist', 'assassin', 'parker']


'The results of actor + woman - man are:\n1. actress\n2. artist\n3. antagonist\n4. assassin\n5. parker\n'

Sentiment and fill-in-the-blank seem to provide unique embeddings; fill-in-the-blank appears to be sharper in the sense that it knows intensity more than sentiment. With the word "terrible," the sentiment embedding gave words with similar negative connotations of the word, but fill-in-the-blank gave words with similar intensity to the word. Mapping the emphasis of the words helps with calculating analogies--for example, actor - man + woman = actress--it's almost like a linear combination of vectors where you subtract the gender of the occupation and add the other gender to get the occupational title for that combination of features. I'm not sure what I was expecting when I did the cheery - light + dark analogy, but words that apparently result include therein, revolutionary, and monica.

## Part 6 - Unit Testing

In [21]:
def test_preprocessing():
    vocab = tokenize_inputs()[3]
    vocab_size = len(vocab)
    assert 70000 <= vocab_size <= 80000, f"Vocabulary size should be in the range [70,000, 80,000], got {vocab_size}"
    word2index = word_to_index(vocab)
    assert vocab[word2index['the']], f"word2index['the'] should return a valid integer index, got {word2index['the']}"

def test_lookup():
    vocab, hidden_size = tokenize_inputs()[3], 100

    # copy weight initialization from sentiment.py
    weights_0_1 = 0.2 * np.random.random((len(vocab), hidden_size)) - 0.1
    x = [2, 5, 9]
    assert np.all(weights_0_1[x].sum(axis=0) == (weights_0_1[2] + weights_0_1[5] + weights_0_1[9])), (
        f"Bundled indices should produce the same sum as split indices, got {weights_0_1[x].sum(axis=0)},"
        f" expected {weights_0_1[2] + weights_0_1[5] + weights_0_1[9]}.")

def test_sentiment():
    labels, vocab, input_dataset, _ = tokenize_inputs()[2:]
    weights_0_1, weights_1_2, _, _ = sentiment(vocab, labels, input_dataset, 100, 0.04, 4, eval_test_size=24700, verbose=False)
    test_correct = 0
    test_start = len(input_dataset) - 100

    for i in range(test_start, len(input_dataset)):
        x, y = input_dataset[i], labels[i]

        layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
        layer_2 = sigmoid(np.dot(layer_1, weights_1_2))

        prediction = layer_2.item()
        test_correct += int((prediction >= 0.5) == y)

    accuracy_testing = test_correct / 100
    assert accuracy_testing > 0.6, f"Test accuracy should be comfortably above chance, expected > 0.6, got {accuracy_testing}"

def test_similarity():
    _, _, labels, vocab, input_dataset, _ = tokenize_inputs()
    weights_0_1, _, word2index, _ = sentiment(vocab, labels, input_dataset, 100, 0.04, 2, eval_test_size=24700, verbose=False)

    words = ["beautiful", "terrible", "actor"]
    for w in words:
        common_scores = similar(weights_0_1, word2index, target=w)
        first_word, first_dist = common_scores[0]
        assert first_word == w and first_dist == 0, f"A word should be closest to itself, expected ({w}, 0), got ({common_scores[0]})"

def test_neg_sampling_shape():
    _, _, _, vocab, input_dataset, _ = tokenize_inputs()
    concatenated = np.concatenate(input_dataset)

    np.random.seed(1)
    np.random.shuffle(input_dataset)  # Trask shuffles the reviews once
    hidden_size, window, neg = 50, 2, 5
    weights_0_1 = (np.random.rand(len(vocab), hidden_size) - 0.5) * 0.2
    weights_1_2 = np.random.rand(len(vocab), hidden_size) * 0  # zeros
    layer_2_target = np.zeros(neg + 1); layer_2_target[0] = 1
    review = input_dataset[42]

    for t in range(len(review)):
        target_samples = [review[t]] + list(
            concatenated[(np.random.rand(neg) * len(concatenated)).astype('int')])
        left = review[max(0, t - window): t]
        right = review[t + 1: min(len(review), t + window)]
        layer_1 = np.mean(weights_0_1[left + right], axis=0)  # (hidden,)
        layer_2 = sigmoid(layer_1.dot(weights_1_2[target_samples].T))  # (6,)
        assert layer_2.shape == (neg+1,), f"Layer 2 has shape (neg+1,), expected {(neg+1,)}, got {layer_2.shape}"
        assert np.all(layer_2_target == np.array([1, 0, 0, 0, 0, 0])), f"Layer 2 target must be [1, 0, 0, 0, 0, 0], got {layer_2_target}"

# Copied from other tests
tests = [name for name in dir() if name.startswith("test_")]
for test_name in sorted(tests):
    test_func = globals()[test_name]
    try:
        test_func()
        print(f" PASS: {test_name}")
    except AssertionError as e:
        print(f" FAIL (AssertionError): {test_name} -- {e}")

TypeError: test_accuracy() missing 4 required positional arguments: 'weights_0_1', 'weights_1_2', 'input_dataset', and 'new_labels'